# Notebook Étudiant - MCP + Airbnb (Colab) — VERSION CORRIGÉE

Mini-agent combinant : votre propre serveur MCP de notes (local), le serveur MCP Airbnb (réel ou provisoire), et un planificateur stub/LLM réel.

**Bugs corrigés dans cette version :**
1. `USE_REAL_AIRBNB` et `USE_REAL_LLM` étaient à `True` par défaut, alors que l'énoncé demande de garder les valeurs par défaut pour un run en mode stub (sans clé, sans npm).
2. La cellule `GITHUB_TOKEN` plantait si aucun secret Colab n'était configuré (même en mode stub).
3. `local_notes_server.py` contenait du code Python invalide (`mcp = ## TODO: ...`) et les décorateurs `@mcp.tool()` manquaient.
4. Le serveur Airbnb "provisoire" (stub) promis par l'énoncé n'existait pas : `--stub-airbnb` n'est pas un vrai fichier.
5. `call_llm` et `answer_with_llm` appelaient **toujours** le vrai LLM Azure, même quand `use_real=False` — le mode stub n'était jamais réellement utilisé.
6. `client.complete(...)` était incomplet (juste un commentaire, aucun argument).
7. Les serveurs étaient lancés via `command="mcp", args=["run", chemin_relatif]`, ce qui est instable ; remplacé par `sys.executable` + chemin du script, plus fiable.

## Installation
À exécuter une fois. `npm` n'est nécessaire que pour le vrai serveur Airbnb (`USE_REAL_AIRBNB = True`).

In [ ]:
# On installe le SDK MCP, nest_asyncio (pour "await" dans le notebook) et requests
!pip install -q mcp nest_asyncio requests
# Bibliothèque cliente pour appeler un vrai LLM via les modèles GitHub (Azure Inference)
!pip install -q azure-ai-inference

# Optionnel : installe le vrai serveur MCP Airbnb (nécessite npm/node)
# Pas indispensable si vous restez en mode stub (USE_REAL_AIRBNB = False)
!npm install -g @openbnb/mcp-server-airbnb

In [ ]:
# ------------------------------------------------------------
# Correctif spécifique à Colab/Jupyter :
# Le flux stdout/stderr du kernel IPython n'a pas de vrai ".fileno()",
# ce qui peut faire planter la création de sous-processus (nos serveurs MCP).
# On "patche" fileno() pour qu'il retourne les descripteurs standards (1 et 2).
# ------------------------------------------------------------
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2 (valeurs standards Unix)
    if self is sys.stderr:
        return 2
    return 1

# On patche la classe pour toutes les instances OutStream
OutStream.fileno = _patched_fileno

# Et on patche aussi les instances déjà créées, explicitement
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Configuration
Ajustez les interrupteurs si besoin. **Par défaut, tout est en mode stub** (aucune clé, aucun npm nécessaire).

In [ ]:

import os
from pathlib import Path

# Jeton utilisé pour un éventuel futur serveur MCP en HTTP (pas utilisé en STDIO)
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")

# CORRIGÉ : on garde les valeurs par défaut (False) pour un run en mode stub,
# comme demandé dans l'énoncé ("Laissez les valeurs par défaut pour la partie stubbed").
USE_REAL_AIRBNB = False  # True si vous avez installé le serveur npm @openbnb/mcp-server-airbnb
USE_REAL_LLM = False     # True si vous avez défini la variable GITHUB_TOKEN


In [ ]:
import os

# On copie l'environnement courant et on y ajoute notre jeton MCP_HTTP_TOKEN.
# BASE_ENV sera transmis à chaque sous-processus serveur (notes / airbnb)
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [ ]:

import os

# CORRIGÉ : cette cellule plantait si aucun secret "GITHUB_TOKEN" n'était
# configuré dans Colab — même en mode stub, où ce jeton n'est pas nécessaire.
# On protège maintenant l'appel avec try/except pour continuer sans erreur.
try:
    from google.colab import userdata  # API des secrets Colab

    # Si votre secret est enregistré sous la clé "GITHUB_TOKEN" dans Colab :
    token = userdata.get("GITHUB_TOKEN")
    if token:
        os.environ["GITHUB_TOKEN"] = token
except Exception:
    # Pas de secret configuré, ou notebook exécuté hors de Colab : on continue
    # simplement en mode stub (aucun jeton nécessaire).
    pass

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))


## Serveur MCP local "notes"

In [ ]:

# ------------------------------------------------------------
# On écrit un petit serveur MCP local dans un fichier séparé
# (local_notes_server.py), qui sera ensuite lancé en sous-processus.
# CORRIGÉ : le code d'origine contenait du texte invalide
# ("mcp = ## TODO: ...") et il manquait les décorateurs @mcp.tool().
# ------------------------------------------------------------
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

# Liste en mémoire pour stocker les notes (remise à zéro à chaque démarrage du serveur)
notes = []

# On crée le serveur MCP et on le nomme "NotesDemo"
mcp = FastMCP("NotesDemo")


# OUTIL : add_note - ajoute une note à la liste en mémoire
@mcp.tool()
def add_note(text: str) -> str:
    \"\"\"Add a note to the in-memory list.\"\"\"
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"


# OUTIL : list_notes - retourne toutes les notes enregistrées, une par ligne
@mcp.tool()
def list_notes() -> str:
    \"\"\"List saved notes.\"\"\"
    if not notes:
        return "No notes yet"
    # CORRIGÉ : on sépare chaque note par un retour à la ligne (\\n),
    # sinon toutes les notes seraient collées ensemble sans séparation
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))


if __name__ == "__main__":
    mcp.run()
'''.strip() + "\n",
    encoding="utf-8",
)
print("wrote", LOCAL_SERVER)


## Serveur MCP "Airbnb" provisoire (stub)

**AJOUTÉ (manquant dans le notebook d'origine)** : l'énoncé promettait *"un serveur Airbnb provisoire (renvoie des annonces fixes) comme solution de secours"*, mais ce serveur n'existait pas — la config par défaut pointait vers un fichier `--stub-airbnb` inexistant. Voici l'implémentation manquante.

In [ ]:

# ------------------------------------------------------------
# Serveur MCP "provisoire" (stub) qui simule le vrai serveur Airbnb.
# Utilisé automatiquement quand USE_REAL_AIRBNB = False.
# Aucune connexion internet ni npm nécessaire.
# ------------------------------------------------------------
AIRBNB_STUB_SERVER = Path("airbnb_stub_server.py")
AIRBNB_STUB_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

# On crée le serveur MCP et on le nomme "AirbnbStub"
mcp = FastMCP("AirbnbStub")

# Annonces fixes (fausses données), utilisées uniquement pour la démo
LISTINGS = {
    "paris": [
        {"title": "Cosy studio near the Eiffel Tower", "price_per_night": 95, "rating": 4.8},
        {"title": "Charming Montmartre apartment", "price_per_night": 110, "rating": 4.6},
    ],
    "london": [
        {"title": "Modern flat in Shoreditch", "price_per_night": 120, "rating": 4.7},
        {"title": "Cosy room near Camden Market", "price_per_night": 80, "rating": 4.5},
    ],
    "new york": [
        {"title": "Brooklyn loft with skyline view", "price_per_night": 150, "rating": 4.9},
        {"title": "Manhattan studio near Central Park", "price_per_night": 200, "rating": 4.7},
    ],
}


# OUTIL : search_listings - retourne quelques annonces fixes pour une ville connue
@mcp.tool()
def search_listings(location: str) -> list:
    \"\"\"Return a few fixed Airbnb-style listings for a supported city.\"\"\"
    key = location.strip().lower()
    return LISTINGS.get(key, [])


if __name__ == "__main__":
    mcp.run()
'''.strip() + "\n",
    encoding="utf-8",
)
print("wrote", AIRBNB_STUB_SERVER)


## Fonctions utilitaires côté client (conversion d'outils, planificateur stub, LLM réel optionnel)

In [ ]:

import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


def convert_tool(tool, prefix: str):
    """Transforme un outil MCP en spécification de fonction pour un LLM,
    en préfixant son nom (ex: "notes__add_note") pour savoir plus tard
    à quel serveur (notes ou airbnb) renvoyer l'appel."""
    # Azure exige le format ^[a-zA-Z0-9_\.-]+$, donc pas de "/" dans le nom
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    AJOUTÉ (fonction manquante dans le notebook d'origine) :
    Planificateur "factice" (stub), qui simule un LLM sans appeler de vrai modèle.
    On analyse le prompt avec des mots-clés simples pour décider quel(s) outil(s)
    appeler, parmi ceux disponibles (préfixés "notes__" ou "airbnb__").
    """
    text = prompt.lower()
    names = {f["function"]["name"] for f in functions}
    calls: List[Dict[str, Any]] = []

    # --- Recherche Airbnb : on cherche une ville connue + un mot-clé de recherche ---
    known_cities = ["paris", "london", "new york"]
    city = next((c for c in known_cities if c in text), None)
    airbnb_tool = next((n for n in names if n.endswith("__search_listings")), None)
    search_keywords = ["stay", "listing", "airbnb", "logement", "find", "search", "séjour", "apartment"]
    if airbnb_tool and city and any(k in text for k in search_keywords):
        calls.append({"name": airbnb_tool, "args": {"location": city.title()}})

    # --- Ajout de note : on cherche un mot-clé de sauvegarde ---
    add_note_tool = next((n for n in names if n.endswith("__add_note")), None)
    note_keywords = ["note", "remember", "save", "enregistre", "sauvegarde"]
    if add_note_tool and any(k in text for k in note_keywords):
        calls.append({"name": add_note_tool, "args": {"text": prompt}})

    # --- Liste des notes : on cherche une demande de consultation ---
    list_notes_tool = next((n for n in names if n.endswith("__list_notes")), None)
    list_keywords = ["list notes", "mes notes", "show notes", "previous notes", "toutes mes notes"]
    if list_notes_tool and any(k in text for k in list_keywords):
        calls.append({"name": list_notes_tool, "args": {}})

    return calls


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Décide quels outils appeler pour répondre au prompt (planifie les tool_calls)."""
    # CORRIGÉ : la fonction d'origine appelait TOUJOURS le vrai LLM Azure,
    # même quand use_real=False. On ajoute maintenant le branchement manquant.
    if not use_real:
        return stub_plan(prompt, functions)

    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")

    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))

    # CORRIGÉ : l'appel était incomplet (juste un commentaire, sans arguments).
    # On envoie le prompt et la liste des outils disponibles au LLM.
    resp = client.complete(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Plan MCP tool calls to help the user."},
            {"role": "user", "content": prompt},
        ],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )

    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:

def stub_answer(user_prompt: str, tool_calls: List[Dict[str, Any]], tool_results: List[Dict[str, Any]]) -> str:
    """
    AJOUTÉ (fonction manquante) : génère une réponse finale lisible SANS appeler
    de LLM, à partir des résultats d'outils déjà obtenus. Utilisé quand
    USE_REAL_LLM = False.
    """
    lines = [f"**Question :** {user_prompt}", ""]

    if not tool_results:
        lines.append("Aucun outil n'a été appelé pour cette question (mode stub).")
    else:
        for r in tool_results:
            lines.append(f"### Résultat de `{r['name']}`")
            for c in r["content"]:
                lines.append(f"- {c}")
            lines.append("")

    lines.append("## Tools used")
    seen = set()
    for r in tool_results:
        if r["name"] not in seen:
            lines.append(f"- {r['name']}")
            seen.add(r["name"])

    return "\n".join(lines)


def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    """Rédige la réponse finale à partir des résultats d'outils."""
    # CORRIGÉ : cette fonction appelait TOUJOURS le vrai LLM Azure, même en
    # mode stub (use_real=False). On ajoute le branchement manquant.
    if not use_real:
        return stub_answer(user_prompt, tool_calls, tool_results)

    import os
    import json

    # On raccourcit les résultats d'outils avant de les envoyer à gpt-4o
    # (pour éviter de dépasser la limite de tokens)
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )

    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestration (connecte les deux serveurs et exécute les tool_calls)

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def orchestrate(prompt: str):
    # CORRIGÉ : on utilise sys.executable (l'interpréteur Python courant) au lieu
    # de "mcp run <chemin relatif>", qui s'est révélé instable (résolution de
    # chemin imprévisible selon le répertoire courant du sous-processus).
    local_params = StdioServerParameters(
        command=sys.executable,
        args=[str(LOCAL_SERVER)],
        env=BASE_ENV,
    )

    # Par défaut (mode stub) : on utilise notre faux serveur Airbnb local
    # CORRIGÉ : le fichier "--stub-airbnb" n'existait pas ; on pointe maintenant
    # vers le vrai fichier AIRBNB_STUB_SERVER défini plus haut.
    airbnb_params = StdioServerParameters(
        command=sys.executable,
        args=[str(AIRBNB_STUB_SERVER)],
        env=BASE_ENV,
    )

    # Si demandé, on utilise le VRAI serveur MCP Airbnb (nécessite npm)
    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )

    # On se connecte au serveur de notes
    async with stdio_client(local_params) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()

            # On se connecte au serveur Airbnb (réel ou stub)
            async with stdio_client(airbnb_params) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = await airbnb_sess.list_tools()

                    # On fusionne les outils des deux serveurs, en les préfixant
                    # pour savoir ensuite où router chaque appel
                    functions = (
                        [convert_tool(t, "notes") for t in local_tools.tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
                    )

                    # Le planificateur (stub ou LLM réel) propose des appels d'outils
                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                    print("tool_calls:", tool_calls)

                    # On exécute chaque appel d'outil, en le routant vers le bon serveur
                    tool_results = []
                    for call in tool_calls:
                        name = call["name"]
                        args = call["args"]
                        # Le nom est de la forme "prefix__tool_name" (ex: "notes__add_note")
                        prefix, tool_name = name.split("__", 1)

                        if prefix == "notes":
                            res = await local_sess.call_tool(tool_name, args)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": [c.text for c in res.content if hasattr(c, "text")],
                                }
                            )
                        elif prefix == "airbnb":
                            res = await airbnb_sess.call_tool(tool_name, args)
                            payload = []
                            if hasattr(res, "content"):
                                for c in res.content:
                                    if hasattr(c, "text"):
                                        payload.append(c.text)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": payload,
                                }
                            )

                    # On génère la réponse finale (stub ou LLM réel) à partir des résultats
                    final_answer = answer_with_llm(
                        user_prompt=prompt,
                        tool_calls=tool_calls,
                        tool_results=tool_results,
                        use_real=USE_REAL_LLM,
                    )

                    return tool_calls, tool_results, final_answer

## Démo
Ajustez le prompt ci-dessous pour tester d'autres villes (paris / london / new york) ou d'autres notes.
Basculez `USE_REAL_AIRBNB` / `USE_REAL_LLM` sur `True` quand vous êtes prêt à utiliser les vrais services.

In [ ]:
# Ce prompt déclenche à la fois une recherche Airbnb (ville "Paris")
# ET l'ajout d'une note, pour démontrer les deux serveurs en une seule requête.
# N'hésitez pas à essayer d'autres villes ("london", "new york") ou d'autres notes.
prompt = "Find Airbnb listings in Paris and save a note that I'm looking at Paris apartments"

# On appelle l'orchestrateur et on affiche les résultats
tool_calls, tool_results, final_answer = await orchestrate(prompt)
print("\ntool_results:", tool_results)
print("\n--- Final Answer ---")
print(final_answer)